# Projeto Seq2Seq para Geração de Descrições de Imagem

Este notebook documenta o comportamento de um projeto que utiliza uma arquitetura seq2seq com atenção para gerar descrições de imagens a partir de um conjunto de labels. O fluxo do projeto é composto pelas seguintes etapas:

1. **Geração de Dados:** Um script gera exemplos sintéticos (e também incorpora exemplos reais) utilizando templates variados para produzir descrições e um conjunto de labels correspondentes.
2. **Manipulação Humana:** Entre a geração e o pré-processamento, há uma intervenção manual para garantir que o dataset seja consistente, removendo duplicados e corrigindo possíveis inconsistências.
3. **Pré-processamento:** Limpeza dos textos, normalização, tokenização dos labels e descrições e divisão em conjuntos de treino e teste.
4. **Treino do Modelo:** Treinamento de um modelo seq2seq com atenção utilizando PyTorch, onde o encoder processa os labels e o decoder gera a descrição.
5. **Inferência:** Uso do modelo treinado para gerar uma descrição dada uma lista de labels.

A seguir, serão apresentados trechos de código e explicações referentes a cada etapa do projeto.

## 1. Geração de Dados

O script `1_generate.py` cria um dataset sintético a partir de um conjunto de categorias (como veículos, pessoas, infraestrutura, etc.) e condições contextuais (tempo, localização, comportamento). Para cada exemplo, uma descrição é gerada utilizando um template em português e os labels associados são extraídos. Ao final, os dados são salvos em um arquivo JSON.

Exemplo de trecho de código (simplificado):

In [ ]:
import json
import random

# Definição de categorias e templates
categories = {
    "veiculos": ["carro", "autocarro", "camião"],
    "pessoas": ["peão", "ciclista"],
    "infraestrutura": ["passadeira", "semáforo"],
    "sinais_transito": ["sinal de stop", "sinal de limite de velocidade", "sinal de passadeira"]
}

weather_conditions = ["céu limpo", "nublado", "chuvoso", "tempestuoso", "com nevoeiro", "vento forte", "com neve"]
time_of_day = ["amanhecer", "anoitecer", "dia", "noite"]
locations = ["residencial", "parque de estacionamento", "túnel", "cidade", "autoestrada"]
object_behaviors = ["a circular rapidamente", "parados no semáforo", "a aguardar para atravessar", "estacionados", "a circular lentamente", "a atravessar a via", "em obras"]

# Seleção aleatória dos itens e geração de exemplos
def generate_synthetic_data(num_samples=1000):
    synthetic_data = []
    for _ in range(num_samples):
        veiculos = random.sample(categories["veiculos"], k=random.randint(1,2))
        pessoas = random.sample(categories["pessoas"], k=random.randint(1,2))
        infraestrutura = random.sample(categories["infraestrutura"], k=random.randint(1,2))
        sinais = random.sample(categories["sinais_transito"], k=random.randint(1,2))
        
        weather = random.choice(weather_conditions)
        time_of_day_choice = random.choice(time_of_day)
        location = random.choice(locations)
        traffic = random.choice(["trânsito leve", "trânsito moderado", "trânsito intenso", "engarrafamento"])
        behavior = random.choice(object_behaviors)
        pessoas_behavior = random.choice(["a aguardar para atravessar", "a caminhar pela via", "à espera junto ao semáforo"])

        # Seleção de um template e formatação da descrição
        templates = [
            "{time}, num dia {weather}, observam-se {veiculos} {behavior}, enquanto {pessoas} estão {pessoas_behavior}, {location}, com {traffic}.",
            "Durante a {time}, sob um clima {weather}, podem ver-se {pessoas} {pessoas_behavior}, bem como {infraestrutura} e {sinais} visíveis {location}, além de {veiculos} {behavior}, com {traffic}."
        ]
        template = random.choice(templates)
        description = template.format(
            veiculos=", ".join(veiculos),
            pessoas=", ".join(pessoas),
            infraestrutura=", ".join(infraestrutura),
            sinais=", ".join(sinais),
            weather=weather,
            time=time_of_day_choice,
            location=location,
            traffic=traffic,
            behavior=behavior,
            pessoas_behavior=pessoas_behavior
        )
        
        # Adicionar tokens de início e fim
        description = f"startseq {description} endseq"
        
        example = {"labels": veiculos + pessoas + infraestrutura + sinais, "description": description, "origem": "sintético"}
        synthetic_data.append(example)
    return synthetic_data

# Gerar e salvar os dados
data = generate_synthetic_data(1000)
with open('data/synthetic_and_real_data.json', 'w', encoding='utf-8') as f:
    json.dump(data, f, indent=4, ensure_ascii=False)
print("Dados sintéticos salvos com sucesso!")

## 2. Manipulação e Pré-processamento dos Dados

Após a geração dos dados, houve uma etapa de verificação manual para garantir a qualidade do dataset. Essa etapa inclui a remoção de duplicados e a verificação de consistência, como evidenciado pelo script `deteta_duplicados.py`.

Em seguida, o script `2_pre_processamento.py` realiza a limpeza dos textos (remoção de acentos, conversão para minúsculas e remoção de caracteres especiais), tokeniza os textos (labels e descrições) utilizando uma implementação customizada (`SimpleTokenizer`) e aplica padding nas sequências. Por fim, os dados são divididos em conjuntos de treino e teste e salvos em arquivos para uso futuro.

In [ ]:
from tokenizer_utils import SimpleTokenizer, pad_sequences
import json
import re
import unicodedata
import numpy as np
from sklearn.model_selection import train_test_split

# Função para limpar o texto
def clean_text(text):
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8', 'ignore')
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# Carregar dataset (exemplo real/sintético)
with open('data/exemplo_dataset.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# Processar descrições e labels
descriptions = []
labels_text = []
for d in data:
    desc = d.get("description", "")
    desc = desc.replace("startseq", "").replace("endseq", "").strip()
    desc_clean = "startseq " + clean_text(desc) + " endseq"
    descriptions.append(desc_clean)
    labs = d.get("labels", [])
    labels_text.append(" ".join([clean_text(l) for l in labs]))

# Criar tokenizadores
label_tokenizer = SimpleTokenizer(oov_token="<UNK>", filters='')
label_tokenizer.fit_on_texts(labels_text)

description_tokenizer = SimpleTokenizer(oov_token="<UNK>", filters='')
description_tokenizer.fit_on_texts(descriptions)

# Converter textos em sequências
label_seq = label_tokenizer.texts_to_sequences(labels_text)
desc_seq = description_tokenizer.texts_to_sequences(descriptions)

# Aplicar padding
max_label_length = min(max(len(seq) for seq in label_seq), 30)
max_desc_length = min(max(len(seq) for seq in desc_seq), 270)
label_padded = pad_sequences(label_seq, maxlen=max_label_length, padding='post')
desc_padded = pad_sequences(desc_seq, maxlen=max_desc_length, padding='post')

# Dividir os dados
label_train, label_test, desc_train, desc_test = train_test_split(label_padded, desc_padded, test_size=0.2, random_state=42)
print("Pré-processamento concluído.")

## 3. Treinamento do Modelo Seq2Seq com Atenção

O script `3_train.py` implementa o treinamento de um modelo seq2seq com atenção (baseado na abordagem Luong). O modelo consiste de um encoder que processa os labels e um decoder que gera a descrição. Durante o treinamento, é utilizado o teacher forcing e a função de perda é a entropia cruzada (com ignorância do índice de padding).

Segue um resumo do pipeline de treinamento:

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

# Exemplo simplificado de definição do dataset
class Seq2SeqDataset(Dataset):
    def __init__(self, encoder_data, decoder_input, decoder_target):
        self.encoder_data = encoder_data
        self.decoder_input = decoder_input
        self.decoder_target = decoder_target
    def __len__(self):
        return len(self.encoder_data)
    def __getitem__(self, idx):
        return (
            torch.LongTensor(self.encoder_data[idx]),
            torch.LongTensor(self.decoder_input[idx]),
            torch.LongTensor(self.decoder_target[idx])
        )

# Definição do modelo (encoder, decoder com atenção e seq2seq) é feita conforme o script
# e o treinamento é realizado iterando sobre o DataLoader e salvando o melhor modelo.

print("Treinamento do modelo (exemplo simplificado) iniciado...")

## 4. Inferência

No script `4_inference.py`, o modelo treinado é carregado e usado para gerar uma descrição a partir de um conjunto de labels fornecido. O processo envolve a tokenização dos labels, passagem pelo encoder e, em seguida, geração da sequência de saída pelo decoder até encontrar o token de fim (`endseq`).

Segue um exemplo de função de inferência:

In [ ]:
import torch
import pickle
from train import Encoder, Decoder, Seq2Seq

# Carregar tokenizadores e modelo
with open('models/label_tokenizer.pkl', 'rb') as f:
    label_tokenizer = pickle.load(f)
with open('models/description_tokenizer.pkl', 'rb') as f:
    description_tokenizer = pickle.load(f)

label_vocab_size = len(label_tokenizer.word_index) + 1
desc_vocab_size = len(description_tokenizer.word_index) + 1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = Encoder(label_vocab_size, emb_dim=300, hidden_dim=256).to(device)
decoder = Decoder(desc_vocab_size, emb_dim=300, enc_hidden_dim=256, dec_hidden_dim=512).to(device)
model = Seq2Seq(encoder, decoder).to(device)
model.load_state_dict(torch.load('models/best_model.pt', map_location=device))
model.eval()

def generate_description(labels):
    # Preprocessamento dos labels
    label_str = " ".join(labels).lower()
    label_seq = label_tokenizer.texts_to_sequences([label_str])
    label_seq = torch.LongTensor(label_seq).to(device)

    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(label_seq)
        input_token = torch.LongTensor([[description_tokenizer.word_index['startseq']]]).to(device)
        output_sentence = []
        for _ in range(270):
            output, hidden, cell = model.decoder(input_token.squeeze(1), hidden, cell, encoder_outputs)
            top1 = output.argmax(-1).item()
            if top1 == description_tokenizer.word_index.get('endseq'):
                break
            word = description_tokenizer.index_word.get(top1, '')
            output_sentence.append(word)
            input_token = torch.LongTensor([[top1]]).to(device)
    return " ".join(output_sentence)

# Exemplo de uso
example_labels = ["céu_limpo", "noite", "parque_de_estacionamento", "sinal_de_stop", "carro", "peao", "passadeira"]
print("Labels:", example_labels)
print("Descrição gerada:", generate_description(example_labels))

## Considerações Finais

Este projeto exemplifica a integração de várias etapas essenciais para o desenvolvimento de um sistema de geração de descrições a partir de labels:

- **Geração de Dados:** Uso de templates e dados sintéticos para criar um grande dataset.
- **Intervenção Humana:** Verificação e remoção de duplicados (e possivelmente outros ajustes) para garantir a qualidade dos dados.
- **Pré-processamento:** Limpeza, tokenização e preparação dos dados para o treinamento.
- **Modelagem:** Implementação de uma arquitetura seq2seq com atenção para gerar descrições de forma condicional.
- **Inferência:** Geração de descrições a partir de novas entradas de labels.

Este notebook serve tanto como documentação quanto como guia prático para execução e experimentação com o projeto.